# Helmet Compliance Detector — Colab Training Notebook

Trains both the YOLOv8 baseline and the YOLOv8+CBAM variant on a free-tier Colab GPU (T4).

This notebook uses the **hybrid workflow**: the dataset was already downloaded and
converted to YOLO format locally (`data/prepare_dataset.py`), zipped, and uploaded to
Google Drive as `data_dataset.zip` — Colab just unzips it, no Kaggle credentials needed
here. Steps: mount Drive -> get project code -> install deps -> unzip dataset -> train
baseline -> train CBAM -> compare.

In [ ]:
!nvidia-smi

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
PROJECT_DIR = '/content/drive/MyDrive/helmet-detection'
import os
os.makedirs(PROJECT_DIR, exist_ok=True)

## Get the project code

Cloned straight from GitHub (public repo, no credentials needed). `PROJECT_DIR` must be
empty the first time you clone into it (Drive already created it above); if you're
re-running this notebook in a later session and the folder already has the repo in it,
skip the clone and just `%cd` in, or `git pull` to update.

In [ ]:
%cd {PROJECT_DIR}
!git clone https://github.com/rozengoza/od.git . 2>/dev/null || (echo 'repo already cloned here, pulling latest instead' && git pull)
!pip install -q -r requirements.txt

## Get the prepared dataset

Upload `data_dataset.zip` (created locally by `python -m zipfile -c data_dataset.zip
data/dataset`) into `PROJECT_DIR` on Drive (same folder as the code), then unzip it here.
This avoids re-downloading from Kaggle inside Colab entirely.

In [ ]:
import os
assert os.path.exists('data_dataset.zip'), \
    'data_dataset.zip not found in PROJECT_DIR — upload it to Drive first.'
!python -m zipfile -e data_dataset.zip .
!ls data/dataset && cat data/dataset/data.yaml

### (Alternative) download fresh from Kaggle instead

Only needed if you didn't prepare the dataset locally. Uncomment and run instead of the
unzip cell above.

In [ ]:
# from google.colab import files
# import os
# os.makedirs('/root/.kaggle', exist_ok=True)
# uploaded = files.upload()  # select kaggle.json
# for fname in uploaded:
#     os.rename(fname, '/root/.kaggle/kaggle.json')
# os.chmod('/root/.kaggle/kaggle.json', 0o600)
# !python data/prepare_dataset.py --out data/dataset --val-frac 0.1 --test-frac 0.1

## Train baseline YOLOv8s

In [ ]:
!python train.py --variant baseline --data data/dataset/data.yaml --model-size s --epochs 60 --imgsz 640 --batch 16

## Train YOLOv8s + CBAM (novel-method variant)

In [ ]:
!python train.py --variant cbam --data data/dataset/data.yaml --model-size s --epochs 60 --imgsz 640 --batch 16

## Compare baseline vs CBAM on the held-out test split

In [ ]:
!python evaluate.py \
  --weights runs/detect/helmet-baseline-yolov8s/weights/best.pt runs/detect/helmet-cbam-yolov8s/weights/best.pt \
  --names baseline cbam \
  --data data/dataset/data.yaml \
  --out docs/results_comparison.csv

## (Optional) Quick sanity check on a sample image

In [ ]:
from ultralytics import YOLO
from models import register_modules  # only needed for the CBAM checkpoint
model = YOLO('runs/detect/helmet-cbam-yolov8s/weights/best.pt')
results = model.predict('data/dataset/test/images', save=True, conf=0.35)
print('Annotated predictions saved under runs/detect/predict*/')

## After training: bring weights back to your local machine

`runs/` lives under Drive's `PROJECT_DIR` already (since you `%cd`'d into it), so both
`best.pt` files persist there automatically. Download
`runs/detect/helmet-baseline-yolov8s/weights/best.pt` and
`runs/detect/helmet-cbam-yolov8s/weights/best.pt` from the Drive web UI (or sync via the
Drive desktop app) into your local project's matching `runs/detect/.../weights/` paths —
that's what `evaluate.py` and `app/streamlit_app.py` expect locally.